# 本地实战P1-02：单头因果Attention

你的P1-01文件已经完成支持范围内的CPU/CUDA验收，接下来只写单头Attention计算核心。
主要练习入口为 [my_attention_template.py](my_attention_template.py)：讲解、手算、A1–A5和命令集中在同一个文件中，可直接填写，不必另建my_attention.py。
本Notebook和 [P1-02教程](../docs/PRACTICE_P1_02_ATTENTION.md) 作可选补充。
Q/K/V直接给定，本关不做Linear投影、RoPE、Cache分配、输出投影或生成。
参考baseline通过不是个人candidate通过。

In [ ]:
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/practice_attention.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.practice_attention import causal_attention, evaluate_attention
from learning.local_practice import write_report
candidate = None
print('当前验收:', 'reference_baseline' if candidate is None else candidate)

## 1. 先不看代码，手算17.5

Q=[0,sqrt(2)*ln(3)]，K位置0=[1,0]、位置1=[0,1]，V为[10]、[20]。
当前Query绝对位置为1，两个Key都可见。缩放后分数[0,ln(3)]，softmax权重[1/4,3/4]。
输出应为17.5。注意D_k=2、D_v=1，输出宽度来自V，不是K位置数。

In [ ]:
q = torch.tensor([[[[0., math.sqrt(2) * math.log(3)]]]])
k = torch.tensor([[[[1., 0.], [0., 1.]]]])
v = torch.tensor([[[[10.], [20.]]]])
actual = causal_attention(q, k, v, start=1)
torch.testing.assert_close(actual, torch.tensor([[[[17.5]]]]))
print('Q/K/V:', tuple(q.shape), tuple(k.shape), tuple(v.shape))
print('输出:', actual.tolist(), tuple(actual.shape))

## 2. 验收：参考基线与自己的文件分开

先运行baseline确认设备与测试链。填好learning/my_attention_template.py后，将candidate改成该路径再执行；另建文件是可选的。
14项包括数值、矩形位置Mask、因果扰动、输入不修改、梯度和参数拒绝，不含性能结论。
若正在边写边测，建议使用教程中的CLI命令，以新进程避免旧import缓存。

In [ ]:
cpu_report = evaluate_attention('cpu', candidate)
cpu_path = Path('logs/practice') / ('p1-02-cpu-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
write_report(cpu_report, cpu_path)
print(cpu_report['implementation_role'], 'passed=', cpu_report['passed'], 'failed=', cpu_report['failed'])
for case in cpu_report['cases']:
    print(case['name'], case['status'], case.get('error', ''))
print('Report:', cpu_path)
assert cpu_report['status'] == 'passed'

In [ ]:
gpu_report = evaluate_attention('cuda', candidate)
gpu_path = Path('logs/practice') / ('p1-02-cuda-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
write_report(gpu_report, gpu_path)
print(gpu_report['implementation_role'], gpu_report['gpu'],
      'passed=', gpu_report['passed'], 'failed=', gpu_report['failed'])
for case in gpu_report['cases']:
    if case['status'] != 'passed':
        print(case['name'], case['error'])
print('Report:', gpu_path)
assert gpu_report['status'] == 'passed'
if candidate is None:
    print('仅参考基线通过；你的my_attention.py尚未验收。')

## 3. 本关你要填写的A1–A5

A1：Q/K点积除sqrt(D_k)。A2：建立Query绝对位置、Key位置。
A3：屏蔽未来为负无穷。A4：沿Key轴softmax。A5：权重矩阵乘V。
保留梯度；不原地修改输入；不额外添加RoPE/投影。

三个故障练习：去缩放、把未来填0、忽略start。每次只改一个，并保存第一个失败case再修复。
记录见 [实战日志](../docs/LOCAL_PRACTICE_LOG.md)。